# Notebook 3: Sliding Window Attention
Inspired by SnapKV (Li et al., 2024)
Tests window sizes W=64, 128, 256, 512, Full.

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import torch
import time
import csv
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "gpt2-medium"
os.makedirs("results/tables", exist_ok=True)

N_LAYERS, N_HEADS, HEAD_DIM = 24, 16, 64

def kv_bytes(size): return 2 * size * N_LAYERS * N_HEADS * HEAD_DIM * 2

window_sizes = [64, 128, 256, 512, None]
seq_len = 512
rows = []

m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

for W in window_sizes:
    label = f"W={W}" if W else "Full"
    full_bytes = kv_bytes(seq_len)
    win_bytes = kv_bytes(W if W else seq_len)
    mem_reduction_pct = round((1 - win_bytes / full_bytes) * 100, 1)

    ids = tok("Hello world", return_tensors="pt")["input_ids"].to("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = m.generate(input_ids=ids, max_new_tokens=100,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    t1 = time.perf_counter()

    row = {
        "window": label, "seq_len": seq_len,
        "theoretical_kv_bytes": win_bytes,
        "mem_reduction_pct": mem_reduction_pct,
        "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 1e6, 2),
        "latency_ms": round((t1 - t0) * 1000, 2),
        "throughput_tok_s": round((out.shape[1] - ids.shape[1]) / (t1 - t0), 2)
    }
    rows.append(row)
    print(row)

del m
torch.cuda.empty_cache()

with open("results/tables/window_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print("Saved window_results.csv")